# 02 — Model Development
Train-Test Split → Baseline Leaderboard → CV Comparison → Ensemble → Optuna Tuning → Save Best Model

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import copy, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate
from sklearn.pipeline        import Pipeline as SkPipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LogisticRegression, SGDClassifier
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import (RandomForestClassifier, GradientBoostingClassifier,
                                     AdaBoostClassifier, BaggingClassifier,
                                     ExtraTreesClassifier, VotingClassifier)
from sklearn.svm             import SVC
from sklearn.metrics         import (make_scorer, accuracy_score, precision_score,
                                     recall_score, f1_score, roc_auc_score,
                                     classification_report)
from xgboost                 import XGBClassifier
from lightgbm                import LGBMClassifier
from imblearn.pipeline       import Pipeline as ImbPipeline
from imblearn.over_sampling  import SMOTE, SMOTENC
from imblearn.combine        import SMOTETomek

RS       = 42
N_SPLITS = 10
SKF      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RS)
KF       = KFold(n_splits=N_SPLITS,           shuffle=True, random_state=RS)

SCORING = {
    "accuracy"   : make_scorer(accuracy_score),
    "precision_1": make_scorer(precision_score, pos_label=1, zero_division=0),
    "recall_1"   : make_scorer(recall_score,    pos_label=1, zero_division=0),
    "f1_1"       : make_scorer(f1_score,        pos_label=1, zero_division=0),
}

In [ ]:
df = pd.read_csv('../data/processed/maternal_health_cleaned.csv')
print(df.shape)
display(df.head())

## Feature Engineering

In [ ]:
# is_fever: binary flag for body temperature above clinical fever threshold
df["Is_Fever"] = (df["Body Temp"] > 99).astype(int)

binary_cols  = [col for col in df.select_dtypes(include=[np.number]).columns
                if df[col].dropna().isin([0, 1]).all()]
numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns
                if col not in binary_cols]

print(f"numerical ({len(numeric_cols)}): {numeric_cols}")
print(f"binary    ({len(binary_cols)}): {binary_cols}")

## Train-Test Split

In [ ]:
feature_cols = numeric_cols + binary_cols
X = df[feature_cols]
y = df["Risk Level"].map({"High": 1, "Low": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RS, stratify=y
)

print(f"train : {X_train.shape[0]} rows  |  class dist: {y_train.value_counts().to_dict()}")
print(f"test  : {X_test.shape[0]}  rows  |  class dist: {y_test.value_counts().to_dict()}")

## Feature Engineering Variants

In [ ]:
# fe-a: all features including body temp
X_train_A, X_test_A = X_train.copy(), X_test.copy()

# fe-b: drop raw body temp, keep is_fever only
X_train_B = X_train.drop(columns=["Body Temp"], errors="ignore").copy()
X_test_B  = X_test.drop(columns=["Body Temp"],  errors="ignore").copy()

FE_VARIANTS = {"FE-A": (X_train_A, X_test_A), "FE-B": (X_train_B, X_test_B)}

for name, (tr, te) in FE_VARIANTS.items():
    print(f"{name}: train {tr.shape}  features: {tr.columns.tolist()}")

## Model Registry

In [ ]:
BASE_MODELS = {
    "Logistic Regression": LogisticRegression(random_state=RS, max_iter=1000),
    "Decision Tree"      : DecisionTreeClassifier(random_state=RS),
    "Random Forest"      : RandomForestClassifier(random_state=RS, n_jobs=-1),
    "Gradient Boosting"  : GradientBoostingClassifier(random_state=RS),
    "XGBoost"            : XGBClassifier(random_state=RS, verbosity=0, eval_metric="logloss", n_jobs=-1),
    "LightGBM"           : LGBMClassifier(random_state=RS, verbose=-1, n_jobs=-1),
    "AdaBoost"           : AdaBoostClassifier(random_state=RS),
    "SVM"                : SVC(random_state=RS, probability=True),
    "Bagging"            : BaggingClassifier(random_state=RS, n_jobs=-1),
    "Extra Trees"        : ExtraTreesClassifier(random_state=RS, n_jobs=-1),
    "SGD"                : SGDClassifier(random_state=RS, loss="log_loss", max_iter=1000),
}
print(f"{len(BASE_MODELS)} models registered: {list(BASE_MODELS.keys())}")

## Helper Functions

In [ ]:
def get_cat_indices(X, max_unique=5):
    return [i for i, col in enumerate(X.columns) if X[col].nunique() < max_unique]


def make_resampler(name, cat_idx):
    if name == "None":       return None
    if name == "SMOTE":      return SMOTE(random_state=RS)
    if name == "SMOTE-Tomek":return SMOTETomek(random_state=RS)
    if name == "SMOTE-NC":
        if not cat_idx:
            print("  [info] SMOTE-NC: no categorical features, falling back to SMOTE.")
            return SMOTE(random_state=RS)
        return SMOTENC(categorical_features=cat_idx, random_state=RS)
    raise ValueError(f"unknown resampler: {name!r}")


def build_pipeline(resampler_name, model, cat_idx):
    clf      = copy.deepcopy(model)
    resampler = make_resampler(resampler_name, cat_idx)
    steps = [("scaler", StandardScaler())]
    if resampler is not None:
        steps.insert(0, ("resampler", resampler))
    steps.append(("clf", clf))
    return ImbPipeline(steps) if resampler is not None else SkPipeline(steps)


def summarise_cv(cv_res, fe, resampler, model):
    return {
        "FE"         : fe,
        "Resampler"  : resampler,
        "Model"      : model,
        "Accuracy"   : round(cv_res["test_accuracy"].mean(),    4),
        "Precision_1": round(cv_res["test_precision_1"].mean(), 4),
        "Recall_1"   : round(cv_res["test_recall_1"].mean(),    4),
        "F1_1"       : round(cv_res["test_f1_1"].mean(),        4),
        "Recall_Std" : round(cv_res["test_recall_1"].std(),     4),
    }

## Main Leaderboard
FE-A/B × {None, SMOTE, SMOTE-Tomek, SMOTE-NC} × 11 Models — Stratified 10-Fold CV

In [ ]:
RESAMPLER_NAMES = ["None", "SMOTE", "SMOTE-Tomek", "SMOTE-NC"]
records = []
total = len(FE_VARIANTS) * len(RESAMPLER_NAMES) * len(BASE_MODELS)
done  = 0

for fe_name, (X_tr, _) in FE_VARIANTS.items():
    cat_idx = get_cat_indices(X_tr)
    print(f"\n{fe_name} | cat_idx: {cat_idx} -> {[X_tr.columns[i] for i in cat_idx]}")
    for res_name in RESAMPLER_NAMES:
        for mdl_name, mdl in BASE_MODELS.items():
            done += 1
            try:
                pipe   = build_pipeline(res_name, mdl, cat_idx)
                cv_res = cross_validate(pipe, X_tr, y_train, cv=SKF,
                                        scoring=SCORING, n_jobs=-1)
                row = summarise_cv(cv_res, fe_name, res_name, mdl_name)
                records.append(row)
                print(f"  [{done:>3}/{total}] {fe_name} | {res_name:<13} | {mdl_name:<22} -> Recall_1={row['Recall_1']:.4f}")
            except Exception as exc:
                print(f"  SKIPPED {fe_name} | {res_name} | {mdl_name}: {exc}")

leaderboard = pd.DataFrame(records).sort_values("Recall_1", ascending=False).reset_index(drop=True)
print("\nTop 10:")
display(leaderboard.head(10))
print("\nBottom 10:")
display(leaderboard.tail(10))

## CV Strategy Comparison: KFold-10 vs StratifiedKFold-10

In [ ]:
cv_records = []
for fe_name, (X_tr, _) in FE_VARIANTS.items():
    cat_idx = get_cat_indices(X_tr)
    for mdl_name, mdl in BASE_MODELS.items():
        for cv_name, cv in [("KFold-10", KF), ("StratifiedKFold-10", SKF)]:
            try:
                pipe   = build_pipeline("None", mdl, cat_idx)
                cv_res = cross_validate(pipe, X_tr, y_train, cv=cv,
                                        scoring=SCORING, n_jobs=-1)
                cv_records.append({
                    "FE"         : fe_name,
                    "Model"      : mdl_name,
                    "CV Strategy": cv_name,
                    "Accuracy"   : round(cv_res["test_accuracy"].mean(),    4),
                    "Precision_1": round(cv_res["test_precision_1"].mean(), 4),
                    "Recall_1"   : round(cv_res["test_recall_1"].mean(),    4),
                    "F1_1"       : round(cv_res["test_f1_1"].mean(),        4),
                })
            except Exception as exc:
                print(f"  SKIPPED {fe_name} | {mdl_name} | {cv_name}: {exc}")

cv_comp_df = pd.DataFrame(cv_records).sort_values(["FE", "Model", "CV Strategy"])
display(cv_comp_df.reset_index(drop=True))

## Ensemble Learning: VotingClassifier + BaggingClassifier

In [ ]:
top3_names = (
    leaderboard.groupby("Model")["Recall_1"]
    .max().sort_values(ascending=False).head(3).index.tolist()
)
best_row    = leaderboard.iloc[0]
best_fe     = best_row["FE"]
best_res    = best_row["Resampler"]
X_tr_ens, X_te_ens = FE_VARIANTS[best_fe]
cat_idx_ens = get_cat_indices(X_tr_ens)

print(f"top-3 models: {top3_names}")
print(f"best FE: {best_fe}  |  best resampler: {best_res}")

estimators_ens = [
    (name.lower().replace(" ", "_"), build_pipeline(best_res, BASE_MODELS[name], cat_idx_ens))
    for name in top3_names
]

ensemble_records = []

for vote_type in ["hard", "soft"]:
    vc = VotingClassifier(estimators=copy.deepcopy(estimators_ens), voting=vote_type)
    vc.fit(X_tr_ens, y_train)
    y_pred = vc.predict(X_te_ens)
    rep = classification_report(y_test, y_pred, output_dict=True)
    ensemble_records.append({
        "Model"      : f"VotingClassifier ({vote_type})",
        "Recall_1"   : round(rep["1"]["recall"],    4),
        "Precision_1": round(rep["1"]["precision"], 4),
        "F1_1"       : round(rep["1"]["f1-score"],  4),
        "Accuracy"   : round(accuracy_score(y_test, y_pred), 4),
    })

# bagging with #1 base estimator pipeline
bag = BaggingClassifier(estimator=copy.deepcopy(estimators_ens[0][1]),
                         n_estimators=50, random_state=RS, n_jobs=-1)
bag.fit(X_tr_ens, y_train)
y_pred_bag = bag.predict(X_te_ens)
rep_bag = classification_report(y_test, y_pred_bag, output_dict=True)
ensemble_records.append({
    "Model"      : f"BaggingClassifier (base={top3_names[0]})",
    "Recall_1"   : round(rep_bag["1"]["recall"],    4),
    "Precision_1": round(rep_bag["1"]["precision"], 4),
    "F1_1"       : round(rep_bag["1"]["f1-score"],  4),
    "Accuracy"   : round(accuracy_score(y_test, y_pred_bag), 4),
})

ensemble_df = pd.DataFrame(ensemble_records).sort_values("Recall_1", ascending=False)
display(ensemble_df)

## Optuna Hyperparameter Tuning (Best Model Only)

In [ ]:
best_mdl_name = leaderboard.iloc[0]["Model"]
best_fe_tune  = leaderboard.iloc[0]["FE"]
best_res_tune = leaderboard.iloc[0]["Resampler"]
X_tr_tune, X_te_tune = FE_VARIANTS[best_fe_tune]
cat_idx_tune = get_cat_indices(X_tr_tune)

print(f"optuna target: {best_mdl_name}  |  FE: {best_fe_tune}  |  resampler: {best_res_tune}")

skf_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)
scorer_recall = make_scorer(recall_score, pos_label=1, zero_division=0)


def build_tuned_model(trial, model_name):
    if model_name == "XGBoost":
        return XGBClassifier(
            n_estimators     =trial.suggest_int("n_estimators",       100, 500),
            max_depth        =trial.suggest_int("max_depth",            3,  10),
            learning_rate    =trial.suggest_float("learning_rate",   0.01, 0.30, log=True),
            subsample        =trial.suggest_float("subsample",        0.60, 1.00),
            colsample_bytree =trial.suggest_float("colsample_bytree", 0.60, 1.00),
            scale_pos_weight =trial.suggest_float("scale_pos_weight", 1.00, 5.00),
            random_state=RS, verbosity=0, eval_metric="logloss", n_jobs=-1,
        )
    if model_name == "LightGBM":
        return LGBMClassifier(
            n_estimators =trial.suggest_int("n_estimators",  100, 500),
            max_depth    =trial.suggest_int("max_depth",       3,  10),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
            class_weight =trial.suggest_categorical("class_weight", ["balanced", None]),
            random_state=RS, verbose=-1, n_jobs=-1,
        )
    if model_name == "Random Forest":
        return RandomForestClassifier(
            n_estimators     =trial.suggest_int("n_estimators",        100, 500),
            max_depth        =trial.suggest_int("max_depth",             3,  20),
            min_samples_split=trial.suggest_int("min_samples_split",     2,  10),
            class_weight     =trial.suggest_categorical("class_weight", ["balanced", None]),
            random_state=RS, n_jobs=-1,
        )
    if model_name == "Extra Trees":
        return ExtraTreesClassifier(
            n_estimators     =trial.suggest_int("n_estimators",        100, 500),
            max_depth        =trial.suggest_int("max_depth",             3,  20),
            min_samples_split=trial.suggest_int("min_samples_split",     2,  10),
            class_weight     =trial.suggest_categorical("class_weight", ["balanced", None]),
            random_state=RS, n_jobs=-1,
        )
    if model_name == "Logistic Regression":
        return LogisticRegression(
            C           =trial.suggest_float("C", 0.01, 10, log=True),
            class_weight=trial.suggest_categorical("class_weight", ["balanced", None]),
            random_state=RS, max_iter=1000,
        )
    if model_name == "SVM":
        return SVC(
            C           =trial.suggest_float("C", 0.01, 10, log=True),
            kernel      =trial.suggest_categorical("kernel", ["rbf", "linear"]),
            class_weight=trial.suggest_categorical("class_weight", ["balanced", None]),
            random_state=RS, probability=True,
        )
    if model_name == "Gradient Boosting":
        return GradientBoostingClassifier(
            n_estimators =trial.suggest_int("n_estimators",  100, 400),
            max_depth    =trial.suggest_int("max_depth",       3,   8),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
            random_state=RS,
        )
    if model_name == "AdaBoost":
        return AdaBoostClassifier(
            n_estimators =trial.suggest_int("n_estimators", 50, 300),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
            random_state=RS,
        )
    if model_name == "Decision Tree":
        return DecisionTreeClassifier(
            max_depth        =trial.suggest_int("max_depth",             3,  20),
            min_samples_split=trial.suggest_int("min_samples_split",     2,  10),
            class_weight     =trial.suggest_categorical("class_weight", ["balanced", None]),
            random_state=RS,
        )
    if model_name == "Bagging":
        return BaggingClassifier(
            n_estimators=trial.suggest_int("n_estimators", 10, 100),
            random_state=RS, n_jobs=-1,
        )
    if model_name == "SGD":
        return SGDClassifier(
            alpha       =trial.suggest_float("alpha", 1e-5, 1.0, log=True),
            class_weight=trial.suggest_categorical("class_weight", ["balanced", None]),
            loss="log_loss", random_state=RS, max_iter=1000,
        )
    raise ValueError(f"no search space defined for: {model_name!r}")


def objective(trial):
    m    = build_tuned_model(trial, best_mdl_name)
    pipe = build_pipeline(best_res_tune, m, cat_idx_tune)
    cv   = cross_validate(pipe, X_tr_tune, y_train,
                           cv=skf_inner, scoring=SCORING, n_jobs=-1)
    return cv["test_recall_1"].mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nbest recall (CV): {study.best_value:.4f}")
print(f"best params: {study.best_params}")

In [ ]:
# refit tuned model on full training data and evaluate on test set
tuned_mdl  = build_tuned_model(optuna.trial.FixedTrial(study.best_params), best_mdl_name)
tuned_pipe = build_pipeline(best_res_tune, tuned_mdl, cat_idx_tune)
tuned_pipe.fit(X_tr_tune, y_train)
y_pred_tuned = tuned_pipe.predict(X_te_tune)

# untuned baseline on same split for fair comparison
untuned_pipe = build_pipeline(best_res_tune, copy.deepcopy(BASE_MODELS[best_mdl_name]), cat_idx_tune)
untuned_pipe.fit(X_tr_tune, y_train)
y_pred_untuned = untuned_pipe.predict(X_te_tune)

tuning_comparison = pd.DataFrame({
    "Untuned": {
        "Recall_1"  : recall_score(y_test, y_pred_untuned, pos_label=1),
        "F1_1"      : f1_score(y_test, y_pred_untuned, pos_label=1),
        "AUC-ROC"   : roc_auc_score(y_test, untuned_pipe.predict_proba(X_te_tune)[:, 1]),
        "Accuracy"  : accuracy_score(y_test, y_pred_untuned),
    },
    "Tuned (Optuna)": {
        "Recall_1"  : recall_score(y_test, y_pred_tuned, pos_label=1),
        "F1_1"      : f1_score(y_test, y_pred_tuned, pos_label=1),
        "AUC-ROC"   : roc_auc_score(y_test, tuned_pipe.predict_proba(X_te_tune)[:, 1]),
        "Accuracy"  : accuracy_score(y_test, y_pred_tuned),
    }
}).T.round(4)

print(f"\ntuning comparison — {best_mdl_name}:")
display(tuning_comparison)

## Save Best Model and Test Data

In [ ]:
import os
os.makedirs("../models", exist_ok=True)

# determine overall best pipeline: compare tuned model against leaderboard top
best_recall_tuned   = recall_score(y_test, y_pred_tuned, pos_label=1)
best_recall_untuned = leaderboard.iloc[0]["Recall_1"]

if best_recall_tuned >= best_recall_untuned:
    best_final_pipe    = tuned_pipe
    best_final_X_train = X_tr_tune
    best_final_X_test  = X_te_tune
    print(f"saving tuned model  (recall={best_recall_tuned:.4f})")
else:
    # refit leaderboard #1 configuration on its split
    row0     = leaderboard.iloc[0]
    Xtr0, Xte0 = FE_VARIANTS[row0["FE"]]
    cat0     = get_cat_indices(Xtr0)
    pipe0    = build_pipeline(row0["Resampler"], copy.deepcopy(BASE_MODELS[row0["Model"]]), cat0)
    pipe0.fit(Xtr0, y_train)
    best_final_pipe    = pipe0
    best_final_X_train = Xtr0
    best_final_X_test  = Xte0
    print(f"saving leaderboard #1 (recall={best_recall_untuned:.4f})")

feature_names_final = best_final_X_train.columns.tolist()

payload = {
    "pipeline"     : best_final_pipe,
    "feature_names": feature_names_final,
    "model_name"   : best_mdl_name,
    "X_test"       : best_final_X_test,
    "y_test"       : y_test,
}

with open("../models/best_model.pkl", "wb") as f:
    pickle.dump(payload, f)

print("saved → ../models/best_model.pkl")
print(f"features ({len(feature_names_final)}): {feature_names_final}")